# Analyse des hôtels et prédiction des tarifs touristiques
Booking.com via Silvia & Óscar / Zenodo. Research snapshot published November 2025 for 24–25 January 2026. Source data and derivative outputs: CC BY-NC-SA 4.0.

In [ ]:
from pathlib import Path
import os, json
if Path.cwd().name == 'notebooks': os.chdir('..')
import pandas as pd
from IPython.display import display, Image
quotes=pd.read_csv('data/processed/hotel_quotes.csv')
metrics=json.loads(Path('reports/metrics.json').read_text())
display(quotes.head())
display(pd.Series(json.loads(Path('reports/data_audit.json').read_text())))

## Préparation et qualité
75 room-count contradictions and 150 multi-room offers are excluded. Remaining quotes concern one room, one night and one infant. Review counts are not review scores; amenities are description mentions.

In [ ]:
assert quotes.rooms.eq(1).all()
assert quotes.nights.eq(1).all()
assert not quotes.canonical_url.str.contains('?',regex=False).any()
display(Image('reports/figures/district_prices.png'))
display(Image('reports/figures/amenity_mentions.png'))

## Évaluation par établissement
All observations for one property stay in one partition. Grid search uses GroupKFold; calibration/test hotels do not fit the selected estimator.

In [ ]:
splits=pd.read_csv('reports/property_splits.csv')
assert splits.groupby('property_id').split.nunique().max()==1
display(pd.DataFrame(metrics['validation']).T)
display(pd.DataFrame(metrics['test']).T)
display(Image('reports/figures/predictions.png'))

## Positionnement
Three KMeans groups are ordered by training-cluster median quoted price. They are not official star ratings; observed price is one of the clustering inputs.

In [ ]:
display(pd.read_csv('reports/cluster_summary.csv'))
display(Image('reports/figures/positioning.png'))

## Statistiques et limites
One aggregated row per property is used for exploratory association tests. One stay date cannot identify seasonality; the test sample is small and selected.

In [ ]:
display(json.loads(Path('reports/statistical_results.json').read_text()))
display(pd.read_csv('reports/importance.csv'))

## Exemple réel et tableau de bord
The fitted model uses the recorded package versions. Open `dashboard/index.html` in a browser for interactive exploration.

In [ ]:
from src.predict import predict
display(predict(pd.read_csv('examples/hotel.csv')))